In [111]:
import pandas as pd

In [ ]:
date_to_load_from = "2026-10-23 12:00:00"
df = pd.read_csv("orders_20261023.csv")

Clean data from csv
deduplicate, flag and the invalid orders, and lowercase the order status

In [151]:
# get the rows I want for the fact table "Orders"
orders = df[["order_id","customer_id", "order_date", "product", "total_amount", "status","country", "updated_at"]]
#make sure the rows are from after the cutoffs
cutoff_time = pd.to_datetime(date_to_load_from)
orders["updated_at"] = pd.to_datetime(orders["updated_at"])
orders = orders[orders["updated_at"] > cutoff_time]
#drop duplicated and keep the last entry
orders = orders.drop_duplicates(subset="order_id", keep="last")
flagged_orders = orders[orders["total_amount"].isna()]
#valid orders are te ones with a valid total amount for order... can change based on business needs
valid_orders = orders[orders["total_amount"].notna()]
#lowercase the status' to make them more uniform for querying later on
valid_orders['status'] = valid_orders['status'].str.lower()
#clean total_amount by stripping the $ 
valid_orders['total_amount'] = valid_orders['total_amount'].str.replace('$','',regex=False).astype(float)
# valid_orders

df['updated_at'].min()


'2026-10-23 00:00:05'

make the database connection

In [143]:
import os
import re

from psycopg.pq import ExecStatus
import pandas as pd
from sqlalchemy import text
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
load_dotenv()

    # Fix the connection string to use the modern async psycopg driver
db_url = re.sub(r'^postgresql:', 'postgresql+psycopg:', os.getenv('DATABASE_URL', ''))
engine = create_engine(db_url, echo=True)

# 2. Open a clean connection block to run a quick test query
with engine.connect() as connection:
    result = connection.execute(text("SELECT 'Connection Successful!'"))
    print(result.scalar())

# 3. Read directly into Pandas (Works perfectly without errors!)
df = pd.read_sql("SELECT * FROM orders LIMIT 5;", con=engine)
print(df)

2026-10-24 20:15:15,743 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-10-24 20:15:15,743 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-24 20:15:15,804 INFO sqlalchemy.engine.Engine select current_schema()
2026-10-24 20:15:15,805 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-24 20:15:15,890 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-10-24 20:15:15,891 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-24 20:15:16,044 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-24 20:15:16,044 INFO sqlalchemy.engine.Engine SELECT 'Connection Successful!'
2026-10-24 20:15:16,045 INFO sqlalchemy.engine.Engine [generated in 0.00096s] {}
Connection Successful!
2026-10-24 20:15:16,104 INFO sqlalchemy.engine.Engine ROLLBACK
2026-10-24 20:15:16,136 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-24 20:15:16,137 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog

put dataframe data in database and count the rows in staging orders table and count rows in df and then perform a merge automatically if counts match up... if not alert for further inspection

In [144]:
import pandas as pd
from sqlalchemy import text

valid_orders.to_sql("orders_temp", con=engine, if_exists="replace", index=False)
query = "SELECT count(order_id) from orders_temp"
df = pd.read_sql(query, con=engine)
local_count = len(valid_orders)

# FIX: Added [0, 0] to extract the primitive integer value from the dataframe
db_count = int(df.iloc[0, 0]) 
print(db_count)

if local_count == db_count:
    merge_query = text("""
    MERGE INTO orders AS target
    USING orders_temp AS source
    ON target.order_id = source.order_id::text
    WHEN MATCHED THEN
        UPDATE SET 
            total_amount = source.total_amount::double precision, 
            status = source.status, 
            updated_at = source.updated_at::timestamp
    WHEN NOT MATCHED THEN
        INSERT (order_id, total_amount, status, updated_at)
        VALUES (
            source.order_id::text, 
            source.total_amount::double precision, 
            source.status, 
            source.updated_at::timestamp
        );
""") 
else:
    print("ALERT")
    print("mismatch with data")
    merge_query = None

if merge_query is not None:
    with engine.begin() as connection:
        connection.execute(merge_query)
        connection.execute(text("drop table orders_temp;"))


2026-10-24 20:15:20,492 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-24 20:15:20,494 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR
2026-10-24 20:15:20,494 INFO sqlalchemy.engine.Engine [cached since 4.358s ago] {'table_name': 'orders_temp', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-10-24 20:15:20,556 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_na

2026-10-24 18:21:03,705 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-24 18:21:03,706 INFO sqlalchemy.engine.Engine 
    MERGE INTO orders AS target
    USING orders_temp AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN
        UPDATE SET 
            total_amount = source.total_amount, 
            status = source.status, 
            updated_at = source.updated_at
    WHEN NOT MATCHED THEN
        INSERT (order_id, total_amount, status, updated_at)
        VALUES (source.order_id, source.total_amount, source.status, source.updated_at);

2026-10-24 18:21:03,707 INFO sqlalchemy.engine.Engine [generated in 0.00070s] {}
2026-10-24 18:21:03,773 INFO sqlalchemy.engine.Engine drop table orders_temp;
2026-10-24 18:21:03,775 INFO sqlalchemy.engine.Engine [generated in 0.00174s] {}
2026-10-24 18:21:03,814 INFO sqlalchemy.engine.Engine COMMIT
